# 1. Prepare the articles and count representation

We use the same 7,332 articles as the modern pipeline. This notebook turns the article bodies
into unigram counts for LDA. Run it in the project's `uv` environment.

Duplicate bodies keep their earliest-dated copy, with source order breaking ties. Missing
authors and sections do not remove an article. The dataset describes news coverage, not public opinion.

This notebook retains the PR’s 100-iteration, default-prior exploration in
`Results/classical_exploration/`. The selected model from `main` uses
`selected_configuration.json` (K=40, 50 iterations, priors 0.1/0.025)
and is rerun with `classical-methods/main.py`. These are different experiments;
the exploration does not replace the selected handoff or its labels.


In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
ROOT = REPO/'final-project/code/classical-methods'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from shared.corpus import CORPUS_DIR, audit_corpus, load_corpus, corpus_hash, digest, write_json
from shared import evaluation as ev
import lda as ld
import representation as rep
import notebook_interpret as interpretation
import notebook_artifacts as artifacts
OUT = ROOT.parent/'modern-methods/Results/classical_exploration'
MODERN = ROOT.parent/'modern-methods/Results/modern'
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
if not (CORPUS_DIR/'corpus.csv').exists():
    audit_corpus(source=REPO/'data/self_driving_articles.csv')
df = load_corpus()
assert len(df) == 7332
assert corpus_hash(df) == 'c73293f76ba4b0a18d685e5724f75b1123a86c9012497fc324090989642a8f99'
assert df.article_id.is_unique
publication_summary = df.groupby('publication').agg(articles=('article_id','size'), first_date=('date','min'), last_date=('date','max'))
publication_summary.to_csv(OUT/'corpus_by_publication.csv')
display(publication_summary)

,articles,first_date,last_date
publication,,,
Axios,360,2017-01-09,2019-12-20
Business Insider,200,2016-02-28,2020-03-21
Buzzfeed News,99,2016-02-29,2019-08-27
CNBC,977,2016-01-01,2020-03-20
CNN,116,2016-01-14,2020-03-12
Economist,84,2016-01-08,2019-10-12
Fox News,23,2017-10-31,2018-08-07
Gizmodo,215,2016-01-04,2019-09-30
Hyperallergic,1,2016-04-28,2016-04-28


## Keep the reserved review articles separate

Modern notebook 01 has already reserved 100 articles. We require that same saved reservation;
we never draw another sample here. These articles still help fit the models, but cannot be
used for model-selection review or topic naming. No article text is displayed in this notebook.

In [3]:
reservation = artifacts.load_reservation(MODERN/'review_sample.json', df)
assert len(reservation['article_ids']) == 100
print(f"Reserved {len(reservation['article_ids'])} articles; reservation hash: {digest(reservation)}")

Reserved 100 articles; reservation hash: 73b997390a820be3064878f5a8a88ce15d7b80760ae830f91f87c0dc579736ce


## Tokenize and build the count matrix

We reuse the existing classical tokenizer and custom stopwords. The shared tokenizer lowercases
words and removes English stopwords. Classical filtering also removes the custom list, numeric-only
tokens, and tokens shorter than three characters. The coherence reference keeps its own shared
vocabulary, so both methods are scored against the same reference articles.

LDA uses **counts**, with `min_df=5` articles and `max_df=0.5`, rather than TF-IDF.
The preparation cache binds the matrix, ordered vocabulary, article IDs, code, and settings.
If filtering leaves an empty article, execution stops and lists its ID; no rows are dropped.

In [4]:
MIN_DF, MAX_DF = 5, 0.5
cache = OUT/'cache/representation'
reused = (cache/'manifest.json').exists()
started = time.perf_counter()
X, vocabulary, preparation = rep.prepare_representation(df, cache, min_df=MIN_DF, max_df=MAX_DF)
execution = dict(cache_reused=reused, elapsed_seconds=time.perf_counter()-started,
                 note='This execution only; original stage timings remain in the preparation manifest.')
write_json(OUT/'preparation_execution.json', execution)
write_json(OUT/'representation_summary.json', dict(summary=preparation['summary'], timings=preparation['timings'],
           environment=preparation['environment'], representation_hash=digest(preparation),
           corpus_hash=corpus_hash(df)))
display(pd.Series(preparation['summary']))
display(pd.Series(preparation['timings'], name='Original measured seconds'))
print('Verified cache reused:', reused)

articles         7332.000000
vocabulary      21528.000000
sparsity            0.989737
empty_rows          0.000000
token_min          56.000000
token_median      227.500000
token_mean        339.561511
token_max       10353.000000
dtype: float64

preprocessing_seconds     1.060019
representation_seconds    0.384485
Name: Original measured seconds, dtype: float64

Verified cache reused: True


## Next steps

Continue with notebook 02 to fit the seven topic counts. The earlier reports described a
20-iteration grid and a final K=40 model. We will reassess the seven settings at 100 iterations,
keeping K=40 as a historical candidate. The old reported metrics are not reused as new results.

Counts and full article evidence remain local. Small summaries, manifests, and figures can be
committed. Cache-loading time is separate from original preprocessing and representation time.